# **Detecção de Fraudes com Keras**

**Objetivo**

Treinar um modelo de Redes Neurais para detecção de fraudes em transações financeiras.

### **1. Introdução**

<p align="justify">
Este projeto de Machine Learning visa abordar um desafio crucial no cenário financeiro contemporâneo: a detecção eficiente de fraudes em transações. A segurança das transações financeiras é de suma importância para proteger usuários e instituições contra atividades fraudulentas que podem resultar em perdas substanciais. Nesse contexto, empregamos a poderosa biblioteca Keras para desenvolver um modelo de detecção de fraudes preciso.

<p align="justify">
Ao utilizar redes neurais profundas e técnicas avançadas de aprendizado profundo proporcionadas pelo Keras, buscamos criar um sistema capaz de identificar padrões complexos e não lineares associados a atividades fraudulentas. A natureza dinâmica e em constante evolução das fraudes financeiras exige abordagens sofisticadas, e a flexibilidade oferecida pelo Keras permite a construção de modelos altamente adaptáveis.

**Fonte:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

In [ ]:
# 1.1 Carregando bibliotecas

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from imblearn.under_sampling import RandomUnderSampler

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
# 1.2 Carregando base de dados

dados = pd.read_csv('/content/drive/MyDrive/Portfólio/Dados/creditcard.csv')

In [ ]:
# Visualizando dados

dados.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## **2. Análise Exploratória de Dados**

In [ ]:
# 2.1 Visualizando informações sobre os dados

dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [ ]:
# 2.2 Verificando distribuição das classes

print("Distribuição das classes:")
print(dados['Class'].value_counts())

Distribuição das classes:
Class
0    284315
1       492
Name: count, dtype: int64


## **3. Pré-Processamento de Dados**

In [ ]:
# 3.1 Separando dados de entrada (features) e as classes

features = dados.drop('Class', axis = 1)
classes = dados['Class']

In [ ]:
# 3.2 Padronizando as features

## 3.2.1 Instânciando objeto
scaler = StandardScaler()

## 3.2.2 Aplicando método
features['Time'] = scaler.fit_transform(features['Time'].values.reshape(-1, 1))
features['Amount'] = scaler.fit_transform(features['Amount'].values.reshape(-1, 1))

In [ ]:
# 3.3 Undersampling: balanceando os dados

## 3.3.1 Instânciando objeto
undersampler = RandomUnderSampler(sampling_strategy = 0.5,
                                  random_state = 0)

## 3.3.2 Aplicando método
features_resampled, classes_resampled = undersampler.fit_resample(features,
                                                                  classes)

In [ ]:
# 3.4 Dividindo os dados em treino e teste

X_treino, X_teste, y_treino, y_teste = train_test_split(features_resampled,
                                                        classes_resampled,
                                                        test_size = 0.3,
                                                        random_state = 42)

In [ ]:
# 3.5 Verificando distribuição dos dados

print(X_treino.shape)
print(y_treino.shape)
print(X_teste.shape)
print(y_teste.shape)

(1033, 30)
(1033,)
(443, 30)
(443,)


## **4. Construção e Treinamento do Modelo de Redes Neurais**

In [ ]:
# 4.1 Criando modelo Neural

modelo = Sequential()

modelo.add(Dense(64, activation = 'relu', input_dim = X_treino.shape[1]))
modelo.add(Dropout(0.2))

modelo.add(Dense(32, activation = 'relu'))
modelo.add(Dropout(0.2))

modelo.add(Dense(1, activation = 'sigmoid'))

In [ ]:
# 4.2 Compilando modelo

modelo.compile(loss = 'binary_crossentropy',
               optimizer = Adam(learning_rate = 0.001),
               metrics = ['accuracy'])

In [ ]:
# 4.3 Treinando o modelo

modelo.fit(X_treino,
           y_treino,
           epochs = 20,
           batch_size = 32,
           verbose = 1)

Epoch 1/20
33/33 [==============================] - 2s 3ms/step - loss: 0.4943 - accuracy: 0.7590
Epoch 2/20
33/33 [==============================] - 0s 2ms/step - loss: 0.2652 - accuracy: 0.9313
Epoch 3/20
33/33 [==============================] - 0s 2ms/step - loss: 0.1848 - accuracy: 0.9409
Epoch 4/20
33/33 [==============================] - 0s 2ms/step - loss: 0.1627 - accuracy: 0.9477
Epoch 5/20
33/33 [==============================] - 0s 2ms/step - loss: 0.1491 - accuracy: 0.9458
Epoch 6/20
33/33 [==============================] - 0s 2ms/step - loss: 0.1351 - accuracy: 0.9477
Epoch 7/20
33/33 [==============================] - 0s 3ms/step - loss: 0.1299 - accuracy: 0.9506
Epoch 8/20
33/33 [==============================] - 0s 3ms/step - loss: 0.1301 - accuracy: 0.9545
Epoch 9/20
33/33 [==============================] - 0s 2ms/step - loss: 0.1220 - accuracy: 0.9526
Epoch 10/20
33/33 [==============================] - 0s 2ms/step - loss: 0.1069 - accuracy: 0.9574
Epoch 11/20
33/33 [

In [ ]:
# 4.4 Aplicando o modelo aos dados de teste

previsoes_probabilidade = modelo.predict(X_teste)
previsoes = np.where(previsoes_probabilidade >= 0.5, 1, 0)

14/14 [==============================] - 0s 2ms/step


## **5. Métricas do Modelo**

In [ ]:
# 5.1 Classification Report

print(classification_report(y_teste, previsoes))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       311
           1       0.96      0.92      0.94       132

    accuracy                           0.97       443
   macro avg       0.96      0.95      0.96       443
weighted avg       0.97      0.97      0.97       443



## **6. Considerações Finais**

<p align="justify">
O classification report reflete um desempenho notável do modelo de detecção de fraudes em transações financeiras, construído com base na biblioteca Keras. As métricas de precision, recall e f1-score fornecem insights valiosos sobre a capacidade do modelo em distinguir transações fraudulentas das legítimas.

<p align="justify">
Observa-se uma precisão excepcionalmente alta para ambas as classes, atingindo 97% para transações legítimas (classe 0) e 95% para transações fraudulentas (classe 1). Isso indica a capacidade do modelo em minimizar falsos positivos e falsos negativos, contribuindo para uma tomada de decisão mais confiável.

<p align="justify">
A análise do recall revela um equilíbrio notável entre as classes, com valores de 98% para transações legítimas e 92% para transações fraudulentas. Essa capacidade de recuperar corretamente instâncias de ambas as classes é crucial para uma detecção eficaz e aborda os riscos associados a transações fraudulentas não identificadas.

<p align="justify">
A acurácia geral do modelo atingiu 96%, demonstrando uma eficiência consistente na classificação de transações. Esta métrica é particularmente relevante em cenários de desequilíbrio de classes, onde a prevalência de transações legítimas pode influenciar a interpretação da acurácia.

<p align="justify">
A média ponderada das métricas (weighted avg) confirma a solidez geral do modelo, com um f1-score de 96%. Essa métrica leva em consideração tanto a precisão quanto o recall, proporcionando uma visão holística do desempenho do modelo.

<p align="justify">
Em conclusão, os resultados apresentados indicam que o modelo Keras atende de maneira eficaz ao desafio de detecção de fraudes em transações financeiras, fornecendo uma ferramenta valiosa para instituições financeiras na proteção contra atividades fraudulentas. Contudo, é sempre recomendável uma monitorização contínua e ajustes, à medida que novos padrões de fraude possam surgir.